# 01 — Análise Exploratória de Dados (EDA)

## Objetivo

Este notebook realiza a análise exploratória das bases `application_record.csv` e `credit_record.csv`, utilizadas no projeto de classificação de risco de crédito.

Os principais objetivos desta etapa são:

- compreender a estrutura e a qualidade dos dados;
- analisar o histórico de pagamentos disponível;
- construir a variável-alvo (`TARGET`);
- investigar o desbalanceamento das classes;
- analisar características numéricas e categóricas dos clientes;
- identificar associações estatísticas relevantes para as etapas posteriores de modelagem.

Nesta etapa, o foco é compreender os dados e seus padrões. As transformações destinadas especificamente aos algoritmos de Machine Learning serão realizadas no notebook de pré-processamento.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import chi2_contingency, mannwhitneyu

RANDOM_STATE = 42

## 1. Carregamento e visão inicial dos dados

O projeto utiliza duas bases:

- `application_record.csv`: contém características cadastrais e socioeconômicas dos clientes;
- `credit_record.csv`: contém o histórico mensal de crédito, identificado pela variável `STATUS`.

Inicialmente, as bases serão carregadas e inspecionadas para compreender suas dimensões, variáveis e estrutura.

In [ ]:
from pathlib import Path

DATA_DIR = Path("../data/raw")

application = pd.read_csv(DATA_DIR / "application_record.csv")
credit = pd.read_csv(DATA_DIR / "credit_record.csv")

print(f'application_record: {application.shape}')
print(f'credit_record: {credit.shape}')

In [ ]:
display(application.head())
display(credit.head())

### Estrutura das bases

A base `application_record` possui uma linha por registro cadastral e reúne características demográficas, familiares, profissionais e financeiras dos clientes.

A base `credit_record` possui múltiplas observações por cliente. A variável `MONTHS_BALANCE` representa os diferentes períodos do histórico de crédito, enquanto `STATUS` registra a situação observada em cada período.

Como as duas bases possuem estruturas diferentes, será necessário identificar os clientes presentes em ambas e consolidar o histórico de crédito em uma variável-alvo por cliente.

In [ ]:
credit['STATUS'].value_counts()

### Interpretação da variável `STATUS`

A variável `STATUS` representa a situação do crédito observada em cada período do histórico:

- `0`: atraso de 1 a 29 dias;
- `1`: atraso de 30 a 59 dias;
- `2`: atraso de 60 a 89 dias;
- `3`: atraso de 90 a 119 dias;
- `4`: atraso de 120 a 149 dias;
- `5`: atraso igual ou superior a 150 dias;
- `C`: crédito quitado ou encerrado;
- `X`: ausência de empréstimo/informação no período.

Para este projeto, será considerado **mau pagador (`TARGET = 1`)** o cliente que apresentar pelo menos uma ocorrência de `STATUS` entre `1` e `5`, ou seja, atraso igual ou superior a 30 dias em algum momento do histórico observado.

Os demais clientes serão classificados como **bom pagador (`TARGET = 0`)**.

É importante observar que `TARGET = 0` não significa necessariamente ausência completa de atraso: clientes que apresentaram apenas `STATUS = 0` (atrasos de 1 a 29 dias) permanecem nessa classe.

In [ ]:
ids_comuns = set(application['ID']).intersection(set(credit['ID']))

print(f"Clientes únicos em application_record: {application['ID'].nunique():,}")
print(f"Clientes únicos em credit_record:      {credit['ID'].nunique():,}")
print(f"Clientes presentes nas duas bases:     {len(ids_comuns):,}")

In [ ]:
# Mantém apenas o histórico dos clientes presentes nas duas bases
credit_comum = credit[credit['ID'].isin(ids_comuns)].copy()

# Identifica clientes que apresentaram atraso de 30 dias ou mais
ids_maus_pagadores = set(
    credit_comum.loc[
        credit_comum['STATUS'].isin(['1', '2', '3', '4', '5']),
        'ID'
    ]
)

# Cria uma observação por cliente
target = pd.DataFrame({'ID': list(ids_comuns)})

# TARGET = 1 se houve atraso >= 30 dias; caso contrário, TARGET = 0
target['TARGET'] = target['ID'].isin(ids_maus_pagadores).astype(int)

target['TARGET'].value_counts()

In [ ]:
target_counts = target['TARGET'].value_counts().sort_index()
target_pct = target['TARGET'].value_counts(normalize=True).sort_index() * 100

print("Distribuição percentual:")
print(target_pct.round(2))

plt.figure(figsize=(7, 5))

bars = plt.bar(
    ['Bom pagador (0)', 'Mau pagador (1)'],
    target_counts
)

plt.title('Distribuição da variável TARGET')
plt.ylabel('Número de clientes')

for bar, pct in zip(bars, target_pct):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f'{int(bar.get_height()):,}\n({pct:.2f}%)',
        ha='center',
        va='bottom'
    )

plt.show()

### Desbalanceamento da variável-alvo

A variável `TARGET` apresenta desbalanceamento relevante: aproximadamente **88,23% dos clientes foram classificados como bons pagadores** e **11,77% como maus pagadores**, segundo o critério adotado.

Esse desbalanceamento deverá ser considerado na etapa de modelagem. Um modelo que simplesmente privilegiasse a classe majoritária poderia apresentar acurácia elevada sem identificar adequadamente os maus pagadores.

Por esse motivo, a avaliação dos modelos não deverá se limitar à acurácia. Métricas como **precision, recall, F1-score, ROC-AUC e PR-AUC** serão consideradas posteriormente.

## 2. Qualidade dos dados

Após a construção da variável-alvo, as informações cadastrais são combinadas com o `TARGET`, mantendo apenas os clientes presentes nas duas bases.

Nesta etapa são avaliados o tamanho da base resultante, a unicidade dos clientes, a presença de registros duplicados e os valores ausentes.

In [ ]:
df = application.merge(target, on='ID', how='inner')

print(f"Dimensão da base: {df.shape}")
print(f"Clientes únicos: {df['ID'].nunique():,}")
print(f"Registros duplicados: {df.duplicated().sum():,}")

In [ ]:
missing = pd.DataFrame({
    'Quantidade': df.isna().sum(),
    'Percentual (%)': (df.isna().mean() * 100).round(2)
})

missing = missing[missing['Quantidade'] > 0].sort_values(
    'Quantidade',
    ascending=False
)

missing

### Investigação de `DAYS_EMPLOYED` e `OCCUPATION_TYPE`

Além dos valores ausentes explícitos, a qualidade dos dados também deve considerar possíveis valores especiais ou anômalos.

Durante a inspeção da variável `DAYS_EMPLOYED`, que representa o tempo de emprego em dias, foi identificado o valor `365243`, incompatível com uma duração realista de vínculo empregatício. A seguir, será investigada a frequência desse valor e sua relação com outras características dos clientes.

In [ ]:
sentinela = df['DAYS_EMPLOYED'] == 365243

print(f"Registros com DAYS_EMPLOYED = 365243: {sentinela.sum():,}")
print(f"Percentual da base: {sentinela.mean() * 100:.2f}%")

df.loc[sentinela, 'NAME_INCOME_TYPE'].value_counts()

In [ ]:
print("OCCUPATION_TYPE nos registros com DAYS_EMPLOYED = 365243:")
display(
    df.loc[sentinela, 'OCCUPATION_TYPE']
      .value_counts(dropna=False)
)

print("\nValores ausentes de OCCUPATION_TYPE por tipo de renda:")
display(
    df.loc[df['OCCUPATION_TYPE'].isna(), 'NAME_INCOME_TYPE']
      .value_counts()
)

### Interpretação

Foram identificados **6.135 registros (16,83%)** com `DAYS_EMPLOYED = 365243`. Esse valor não representa literalmente o tempo de emprego e está associado aos clientes classificados como pensionistas, indicando uma codificação especial da base.

Também foram observados **11.323 valores ausentes em `OCCUPATION_TYPE` (31,06%)**. Embora os pensionistas representem parcela relevante dessas ausências, elas também ocorrem em outras categorias de renda.

Esses registros não serão excluídos nesta etapa. O tratamento de `DAYS_EMPLOYED` e `OCCUPATION_TYPE` será definido no notebook de pré-processamento, preservando nesta etapa a função exploratória da análise.

## 3. Análise exploratória das variáveis numéricas

Para facilitar a interpretação das variáveis temporais durante a análise exploratória, serão criadas duas variáveis auxiliares:

- `AGE_YEARS`: idade aproximada do cliente em anos;
- `YEARS_EMPLOYED`: tempo aproximado de emprego em anos.

A codificação especial `365243` de `DAYS_EMPLOYED` será representada como valor ausente apenas na variável auxiliar utilizada nesta análise.

Essas transformações possuem finalidade exploratória. O tratamento definitivo das variáveis para modelagem será realizado no notebook de pré-processamento.

In [ ]:
df_eda = df.copy()

df_eda['AGE_YEARS'] = -df_eda['DAYS_BIRTH'] / 365.25

df_eda['YEARS_EMPLOYED'] = np.where(
    df_eda['DAYS_EMPLOYED'] == 365243,
    np.nan,
    -df_eda['DAYS_EMPLOYED'] / 365.25
)

df_eda[['AGE_YEARS', 'YEARS_EMPLOYED']].describe().round(2)

In [ ]:
numeric_vars = [
    'AMT_INCOME_TOTAL',
    'AGE_YEARS',
    'YEARS_EMPLOYED',
    'CNT_CHILDREN',
    'CNT_FAM_MEMBERS'
]

df_eda[numeric_vars].describe().T.round(2)

### Síntese descritiva

A base apresenta clientes com idade média próxima de 44 anos. Entre os registros com tempo de emprego válido, a média é de aproximadamente 7 anos.

A renda apresenta dispersão considerável, com diferença relevante entre os valores centrais e o máximo observado. As variáveis de número de filhos e tamanho da família também apresentam valores máximos muito superiores às respectivas medianas.

Essas características justificam uma investigação específica da distribuição e de possíveis valores extremos, sem assumir previamente que observações extremas sejam necessariamente erros.

### Distribuição e valores extremos da renda

A renda (`AMT_INCOME_TOTAL`) apresenta média superior à mediana e valor máximo consideravelmente distante dos quartis centrais, indicando uma distribuição assimétrica e a presença de valores extremos.

Para visualizar esse comportamento, será utilizado um boxplot. Nesta etapa, valores extremos serão tratados como observações a serem investigadas, e não automaticamente como erros.

In [ ]:
plt.figure(figsize=(9, 4))

plt.boxplot(
    df_eda['AMT_INCOME_TOTAL'],
    vert=False
)

plt.title('Distribuição da renda total')
plt.xlabel('Renda total')

plt.show()

In [ ]:
q1 = df_eda['AMT_INCOME_TOTAL'].quantile(0.25)
q3 = df_eda['AMT_INCOME_TOTAL'].quantile(0.75)
iqr = q3 - q1

limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

outliers_renda = df_eda[
    (df_eda['AMT_INCOME_TOTAL'] < limite_inferior) |
    (df_eda['AMT_INCOME_TOTAL'] > limite_superior)
]

print(f"Q1: {q1:,.2f}")
print(f"Q3: {q3:,.2f}")
print(f"IQR: {iqr:,.2f}")
print(f"Limite superior pelo critério de 1,5 × IQR: {limite_superior:,.2f}")
print(f"Observações além dos limites: {len(outliers_renda):,}")
print(f"Percentual da base: {len(outliers_renda) / len(df_eda) * 100:.2f}%")

### Interpretação dos valores extremos

Pelo critério de 1,5 vezes o intervalo interquartil (IQR), valores de renda acima de **R$ 380.250** são identificados como estatisticamente extremos. Foram encontradas **1.529 observações (4,19% da base)** além dos limites calculados.

A presença desses valores confirma a assimetria da distribuição de renda, mas não constitui, por si só, evidência de erro nos dados. Rendas elevadas são plausíveis no contexto da variável analisada e, por esse motivo, essas observações não serão removidas automaticamente.

Eventuais transformações ou tratamentos necessários para a modelagem serão avaliados posteriormente no notebook de pré-processamento.

In [ ]:
resumo_target = (
    df_eda
    .groupby('TARGET')[numeric_vars]
    .agg(['mean', 'median'])
    .round(2)
)

resumo_target

### Comparação entre as classes

As estatísticas descritivas indicam algumas diferenças entre os grupos. Os clientes classificados como maus pagadores apresentam, em média, idade ligeiramente menor e renda um pouco superior à dos bons pagadores.

Entretanto, as diferenças observadas são pequenas em relação à dispersão existente dentro de cada grupo. Para número de filhos, tamanho da família e tempo de emprego, as medidas centrais são bastante semelhantes.

Essas comparações são descritivas e não permitem concluir, isoladamente, que essas características expliquem o comportamento de pagamento. A significância e a magnitude das associações serão avaliadas posteriormente por testes estatísticos.

### Taxa de maus pagadores por faixas

As médias e medianas podem ocultar relações não lineares. Por esse motivo, idade, renda e tempo de emprego serão agrupados em faixas exclusivamente para fins exploratórios.

Será calculada a proporção de `TARGET = 1` em cada grupo, permitindo observar se o risco apresenta algum padrão ao longo dessas características.

In [ ]:
df_eda['AGE_GROUP'] = pd.cut(
    df_eda['AGE_YEARS'],
    bins=[20, 30, 40, 50, 60, 70],
    right=False,
    labels=['20–29', '30–39', '40–49', '50–59', '60–69']
)

idade_target = (
    df_eda
    .groupby('AGE_GROUP', observed=False)['TARGET']
    .agg(['count', 'sum', 'mean'])
)

idade_target['Taxa de maus pagadores (%)'] = idade_target['mean'] * 100

idade_target[['count', 'sum', 'Taxa de maus pagadores (%)']].round(2)

In [ ]:
taxa_idade = idade_target['Taxa de maus pagadores (%)']

plt.figure(figsize=(8, 5))

bars = plt.bar(taxa_idade.index.astype(str), taxa_idade)

plt.title('Taxa de maus pagadores por faixa etária')
plt.xlabel('Faixa etária')
plt.ylabel('Maus pagadores (%)')

for bar, valor in zip(bars, taxa_idade):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f'{valor:.2f}%',
        ha='center',
        va='bottom'
    )

plt.show()

### Interpretação da idade

Observa-se uma tendência geral de redução na proporção de maus pagadores conforme aumenta a faixa etária. Os clientes mais jovens apresentam a maior taxa, enquanto o grupo de maior idade apresenta a menor.

A diferença, entretanto, é moderada. A idade pode contribuir para a discriminação entre as classes, mas não aparenta ser suficiente, isoladamente, para determinar o comportamento de pagamento.

### Renda e taxa de maus pagadores

Para investigar a relação entre renda e comportamento de pagamento sem depender apenas da média, os clientes serão divididos em cinco grupos de renda com tamanhos aproximadamente semelhantes.

Essa abordagem permite verificar se a proporção de maus pagadores aumenta ou diminui de maneira consistente conforme a renda.

In [ ]:
df_eda['INCOME_GROUP'] = pd.qcut(
    df_eda['AMT_INCOME_TOTAL'],
    q=5,
    duplicates='drop'
)

renda_target = (
    df_eda
    .groupby('INCOME_GROUP', observed=False)
    .agg(
        Clientes=('TARGET', 'size'),
        Maus_pagadores=('TARGET', 'sum'),
        Taxa_maus_pagadores=('TARGET', 'mean'),
        Renda_mediana=('AMT_INCOME_TOTAL', 'median')
    )
)

renda_target['Taxa_maus_pagadores'] *= 100

renda_target.round(2)

In [ ]:
taxa_renda = renda_target['Taxa_maus_pagadores']
rotulos_renda = [
    f"R$ {valor/1000:.1f} mil"
    for valor in renda_target['Renda_mediana']
]

plt.figure(figsize=(9, 5))

bars = plt.bar(rotulos_renda, taxa_renda)

plt.title('Taxa de maus pagadores por grupo de renda')
plt.xlabel('Renda mediana do grupo')
plt.ylabel('Maus pagadores (%)')

for bar, valor in zip(bars, taxa_renda):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f'{valor:.2f}%',
        ha='center',
        va='bottom'
    )

plt.show()

### Interpretação da renda

Não foi observada uma relação monotônica entre renda e taxa de maus pagadores. A proporção de `TARGET = 1` oscila entre os grupos, em vez de aumentar ou diminuir continuamente conforme a renda.

O grupo intermediário apresentou a menor taxa, enquanto grupos de renda mais elevada apresentaram taxas ligeiramente superiores em alguns casos.

Portanto, embora existam diferenças entre os grupos, a renda isoladamente não apresenta um padrão simples de associação com o comportamento de pagamento.

### Tempo de emprego e taxa de maus pagadores

O tempo de emprego também será agrupado em faixas para investigar possíveis diferenças na proporção de maus pagadores.

Os registros originalmente codificados com `DAYS_EMPLOYED = 365243` não participam desta comparação, pois foram representados como ausentes na variável auxiliar `YEARS_EMPLOYED`.

In [ ]:
df_eda['EMPLOYMENT_GROUP'] = pd.cut(
    df_eda['YEARS_EMPLOYED'],
    bins=[0, 2, 5, 10, 20, np.inf],
    labels=['Até 2 anos', '2–5 anos', '5–10 anos', '10–20 anos', '20+ anos'],
    include_lowest=True
)

emprego_target = (
    df_eda
    .groupby('EMPLOYMENT_GROUP', observed=False)['TARGET']
    .agg(['count', 'sum', 'mean'])
)

emprego_target['Taxa de maus pagadores (%)'] = emprego_target['mean'] * 100

emprego_target[
    ['count', 'sum', 'Taxa de maus pagadores (%)']
].round(2)

### Interpretação do tempo de emprego

As taxas de maus pagadores variam pouco entre as diferentes faixas de tempo de emprego, permanecendo aproximadamente entre 11% e 13%.

Não se observa uma tendência consistente de aumento ou redução da taxa de maus pagadores conforme cresce o tempo de emprego. Assim, de forma isolada, essa variável não apresenta uma relação simples com o comportamento de pagamento.

### Correlação entre variáveis numéricas

A matriz de correlação permite avaliar a intensidade e a direção das relações lineares entre as principais variáveis numéricas.

Essa análise também auxilia na identificação de possíveis redundâncias entre atributos. Correlação, entretanto, não implica causalidade e relações não lineares podem não ser capturadas por esse coeficiente.

In [ ]:
corr_vars = [
    'AMT_INCOME_TOTAL',
    'AGE_YEARS',
    'YEARS_EMPLOYED',
    'CNT_CHILDREN',
    'CNT_FAM_MEMBERS',
    'TARGET'
]

correlacao = df_eda[corr_vars].corr()

fig, ax = plt.subplots(figsize=(9, 7))

im = ax.imshow(correlacao, vmin=-1, vmax=1)

ax.set_xticks(range(len(corr_vars)))
ax.set_yticks(range(len(corr_vars)))
ax.set_xticklabels(corr_vars, rotation=45, ha='right')
ax.set_yticklabels(corr_vars)

for i in range(len(corr_vars)):
    for j in range(len(corr_vars)):
        ax.text(
            j, i,
            f'{correlacao.iloc[i, j]:.2f}',
            ha='center',
            va='center'
        )

plt.colorbar(im, ax=ax, label='Correlação')
plt.title('Matriz de correlação das variáveis numéricas')
plt.tight_layout()
plt.show()

### Interpretação das correlações

A maior correlação observada ocorre entre `CNT_CHILDREN` e `CNT_FAM_MEMBERS` (**0,89**), resultado esperado, pois o número de filhos contribui diretamente para o tamanho da família. Essa associação deverá ser considerada posteriormente na etapa de modelagem, por indicar possível redundância entre as duas variáveis.

Também se observa correlação positiva moderada entre idade e tempo de emprego (**0,34**) e correlações negativas moderadas entre idade e número de filhos (**−0,34**) e tamanho da família (**−0,30**).

Em relação ao `TARGET`, todas as correlações lineares com as variáveis numéricas analisadas são muito próximas de zero. Isso indica ausência de relações lineares fortes individualmente, mas não exclui possíveis relações não lineares, interações entre variáveis ou capacidade preditiva quando os atributos são considerados conjuntamente.

## 4. Análise exploratória das variáveis categóricas

As variáveis categóricas serão analisadas comparando a proporção de maus pagadores (`TARGET = 1`) entre suas diferentes categorias.

O objetivo é identificar diferenças potencialmente relevantes sem interpretar essas associações como relações causais.

In [ ]:
def analisar_categorica(df, coluna):
    resultado = (
        df.groupby(coluna, dropna=False)['TARGET']
        .agg(['count', 'sum', 'mean'])
        .rename(columns={
            'count': 'Clientes',
            'sum': 'Maus_pagadores',
            'mean': 'Taxa_maus_pagadores'
        })
    )

    resultado['Taxa_maus_pagadores'] *= 100

    return resultado.sort_values(
        'Taxa_maus_pagadores',
        ascending=False
    ).round(2)

In [ ]:
variaveis_categoricas_basicas = [
    'CODE_GENDER',
    'FLAG_OWN_CAR',
    'FLAG_OWN_REALTY',
    'NAME_FAMILY_STATUS',
    'FLAG_WORK_PHONE',
    'FLAG_PHONE',
    'FLAG_EMAIL'
]

for coluna in variaveis_categoricas_basicas:
    print(f'\n--- {coluna} ---')
    display(analisar_categorica(df_eda, coluna))

### Características cadastrais básicas

As taxas de maus pagadores apresentam algumas diferenças entre categorias. Clientes sem imóvel próprio, por exemplo, apresentam taxa de 13,06%, frente a 11,14% entre aqueles que possuem imóvel. Também são observadas diferenças segundo gênero, estado civil e presença de e-mail cadastrado.

Em contraste, algumas variáveis apresentam taxas praticamente idênticas entre as categorias. `FLAG_WORK_PHONE`, por exemplo, registra 11,77% para clientes sem telefone comercial e 11,76% para clientes com esse indicador.

Essas diferenças são apenas descritivas. Posteriormente serão utilizados testes estatísticos e medidas de associação para avaliar se os padrões observados possuem evidência estatística e qual é a magnitude dessas associações.

In [ ]:
variaveis_categoricas_detalhadas = [
    'NAME_INCOME_TYPE',
    'NAME_EDUCATION_TYPE',
    'NAME_HOUSING_TYPE',
    'OCCUPATION_TYPE'
]

for coluna in variaveis_categoricas_detalhadas:
    print(f'\n--- {coluna} ---')
    display(analisar_categorica(df_eda, coluna))

### Categorias com múltiplos níveis

As taxas de maus pagadores variam entre categorias de renda, escolaridade, moradia e ocupação. Entretanto, algumas das maiores taxas são observadas em grupos com poucas observações.

Por exemplo, `Academic degree` apresenta taxa elevada, mas contém apenas 32 clientes. De forma semelhante, algumas ocupações com taxas altas possuem amostras relativamente pequenas.

Por esse motivo, as taxas percentuais não devem ser interpretadas isoladamente. Além do tamanho de cada grupo, é necessário avaliar estatisticamente se existe associação entre essas variáveis e o `TARGET` e, principalmente, qual é a magnitude dessa associação.

## 5. Testes estatísticos

### Variáveis categóricas: Qui-quadrado e V de Cramér

Para avaliar a associação entre as variáveis categóricas e o `TARGET`, será aplicado o teste de independência do qui-quadrado.

Como o valor-p informa sobre evidência estatística, mas não sobre a magnitude da associação, também será calculado o **V de Cramér** como medida de tamanho de efeito.

Em uma base relativamente grande, diferenças pequenas podem apresentar significância estatística. Por isso, a interpretação considerará conjuntamente o valor-p e o V de Cramér.

In [ ]:
categorical_vars = [
    'CODE_GENDER',
    'FLAG_OWN_CAR',
    'FLAG_OWN_REALTY',
    'NAME_INCOME_TYPE',
    'NAME_EDUCATION_TYPE',
    'NAME_FAMILY_STATUS',
    'NAME_HOUSING_TYPE',
    'FLAG_WORK_PHONE',
    'FLAG_PHONE',
    'FLAG_EMAIL',
    'OCCUPATION_TYPE'
]

resultados_cat = []

for coluna in categorical_vars:

    tabela = pd.crosstab(
        df_eda[coluna].fillna('Missing'),
        df_eda['TARGET']
    )

    chi2, p, _, _ = chi2_contingency(tabela)

    n = tabela.values.sum()
    r, k = tabela.shape

    cramer_v = np.sqrt(
        chi2 / (n * min(k - 1, r - 1))
    )

    resultados_cat.append({
        'Variável': coluna,
        'Qui-quadrado': chi2,
        'p-valor': p,
        'V de Cramér': cramer_v
    })

resultados_cat = (
    pd.DataFrame(resultados_cat)
    .sort_values('V de Cramér', ascending=False)
)

resultados_cat

In [ ]:
resultados_cat_formatado = resultados_cat.copy()

resultados_cat_formatado['Qui-quadrado'] = (
    resultados_cat_formatado['Qui-quadrado'].round(3)
)

resultados_cat_formatado['p-valor'] = (
    resultados_cat_formatado['p-valor']
    .apply(lambda x: f'{x:.3e}')
)

resultados_cat_formatado['V de Cramér'] = (
    resultados_cat_formatado['V de Cramér'].round(4)
)

resultados_cat_formatado

### Interpretação das associações categóricas

Diversas variáveis categóricas apresentam associação estatisticamente significativa com o `TARGET` ao nível de 5%, incluindo ocupação, posse de imóvel, tipo de renda, gênero, escolaridade, presença de e-mail, estado civil, tipo de moradia e posse de automóvel.

Entretanto, os valores do **V de Cramér são baixos em todas as variáveis analisadas**. A maior associação foi observada em `OCCUPATION_TYPE`, com V de Cramér de aproximadamente **0,042**, seguida por `FLAG_OWN_REALTY` (**0,028**) e `NAME_INCOME_TYPE` (**0,024**).

Isso indica que, apesar da existência de diferenças estatisticamente detectáveis, a magnitude das associações individuais é pequena.

`FLAG_PHONE` e, principalmente, `FLAG_WORK_PHONE` não apresentaram evidência estatística relevante de associação com o `TARGET`.

Esses resultados reforçam a hipótese de que nenhuma característica categórica isolada é suficiente para discriminar fortemente bons e maus pagadores, justificando posteriormente uma abordagem multivariada de Machine Learning.

### Variáveis numéricas: teste de Mann–Whitney

Para comparar as distribuições das variáveis numéricas entre bons e maus pagadores será utilizado o teste não paramétrico de Mann–Whitney.

A escolha desse teste evita assumir normalidade das distribuições, característica particularmente relevante para variáveis como renda, que apresentou assimetria e valores extremos.

Além do valor-p, será calculada uma medida de tamanho de efeito baseada nos postos, permitindo avaliar a magnitude das diferenças observadas.

In [ ]:
from scipy.stats import mannwhitneyu

resultados_num = []

for coluna in numeric_vars:

    grupo_0 = df_eda.loc[
        df_eda['TARGET'] == 0, coluna
    ].dropna()

    grupo_1 = df_eda.loc[
        df_eda['TARGET'] == 1, coluna
    ].dropna()

    u_stat, p = mannwhitneyu(
        grupo_0,
        grupo_1,
        alternative='two-sided'
    )

    n0 = len(grupo_0)
    n1 = len(grupo_1)

    rank_biserial = 1 - (2 * u_stat) / (n0 * n1)

    resultados_num.append({
        'Variável': coluna,
        'Mediana TARGET 0': grupo_0.median(),
        'Mediana TARGET 1': grupo_1.median(),
        'p-valor': p,
        'Efeito rank-biserial': rank_biserial
    })

resultados_num = pd.DataFrame(resultados_num)

resultados_num

In [ ]:
resultados_num_formatado = resultados_num.copy()

resultados_num_formatado['Mediana TARGET 0'] = (
    resultados_num_formatado['Mediana TARGET 0'].round(2)
)

resultados_num_formatado['Mediana TARGET 1'] = (
    resultados_num_formatado['Mediana TARGET 1'].round(2)
)

resultados_num_formatado['p-valor'] = (
    resultados_num_formatado['p-valor']
    .apply(lambda x: f'{x:.3e}')
)

resultados_num_formatado['Efeito rank-biserial'] = (
    resultados_num_formatado['Efeito rank-biserial'].round(4)
)

resultados_num_formatado

### Interpretação das variáveis numéricas

Os testes de Mann–Whitney identificaram diferenças estatisticamente significativas para **idade** e **renda** entre bons e maus pagadores.

A idade apresentou medianas de aproximadamente **42,78 anos** para bons pagadores e **41,54 anos** para maus pagadores (`p < 0,001`). A renda apresentou medianas de **R$ 157.500** e **R$ 171.000**, respectivamente (`p ≈ 0,0017`).

Entretanto, os tamanhos de efeito rank-biserial foram muito pequenos: aproximadamente **0,053 em valor absoluto para idade** e **0,029 para renda**. Isso indica que, apesar da significância estatística, as diferenças entre os grupos possuem baixa magnitude.

Para tempo de emprego, número de filhos e tamanho da família, não foram encontradas diferenças estatisticamente significativas ao nível de 5%.

Assim como observado nas variáveis categóricas, os resultados indicam que nenhuma das características numéricas analisadas apresenta, isoladamente, forte capacidade de discriminação entre bons e maus pagadores.

## 6. Principais conclusões da EDA

A análise exploratória permitiu compreender a estrutura das duas bases e consolidar uma população de **36.457 clientes** presentes simultaneamente nos dados cadastrais e no histórico de crédito.

A variável-alvo foi definida como `TARGET = 1` quando o cliente apresentou pelo menos uma ocorrência de atraso igual ou superior a 30 dias no histórico observado. Segundo esse critério, **11,77% dos clientes foram classificados como maus pagadores**, evidenciando um problema de classificação desbalanceado.

### Qualidade dos dados

Não foram identificados registros duplicados na base consolidada. A variável `OCCUPATION_TYPE` apresenta **11.323 valores ausentes (31,06%)**.

Também foi identificada uma codificação especial em `DAYS_EMPLOYED`: **6.135 registros (16,83%)** possuem o valor `365243`, associado aos clientes pensionistas. O tratamento dessas características será realizado na etapa de pré-processamento.

A distribuição da renda apresentou assimetria e valores extremos. Pelo critério de 1,5 × IQR, **1.529 observações (4,19%)** foram identificadas além dos limites calculados. Esses registros não foram considerados automaticamente como erros e foram preservados para avaliação posterior.

### Relação das características com o TARGET

A análise por faixas indicou uma tendência geral de redução da taxa de maus pagadores com o aumento da idade. Para renda e tempo de emprego, não foi identificado um comportamento monotônico claro.

As correlações lineares entre as variáveis numéricas e o `TARGET` foram muito próximas de zero. Entre os próprios atributos, destacou-se a forte correlação entre `CNT_CHILDREN` e `CNT_FAM_MEMBERS` (**0,89**).

Diversas variáveis categóricas apresentaram associação estatisticamente significativa com o `TARGET`. Entretanto, todos os valores de V de Cramér foram baixos, sendo o maior aproximadamente **0,042** para `OCCUPATION_TYPE`.

Entre as variáveis numéricas, idade e renda apresentaram diferenças estatisticamente significativas entre as classes, mas com tamanhos de efeito muito pequenos. Tempo de emprego, número de filhos e tamanho da família não apresentaram diferenças estatisticamente significativas ao nível de 5%.

### Implicações para a modelagem

De forma geral, nenhuma variável analisada individualmente demonstrou forte capacidade de discriminação entre bons e maus pagadores.

Os resultados indicam que a etapa de modelagem deverá explorar **múltiplas características simultaneamente**, incluindo possíveis relações não lineares e interações entre atributos.

O desbalanceamento do `TARGET` também deverá ser considerado na escolha dos algoritmos e, principalmente, das métricas de avaliação, evitando que a qualidade dos modelos seja julgada apenas pela acurácia.

O tratamento dos valores ausentes, da codificação especial de `DAYS_EMPLOYED` e a preparação das variáveis para os algoritmos serão realizados no notebook seguinte, dedicado ao pré-processamento.